In [1]:
# Data Ingestion, Dataset Inspection and Data Quality Validation

# Import required libraries
import os
import warnings
import shutil
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# Configure pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", "{:.2f}".format)


# Define project folder paths
current_directory = os.getcwd()

# Locate the project folder automatically
if os.path.basename(current_directory).lower() == "notebooks":
    project_folder = os.path.dirname(current_directory)
else:
    project_folder = current_directory

data_folder = os.path.join(project_folder, "data")
raw_data_folder = os.path.join(data_folder, "raw")
processed_data_folder = os.path.join(data_folder, "processed")
output_folder = os.path.join(project_folder, "outputs")
table_folder = os.path.join(output_folder, "tables")

# Create output folders if they do not exist
os.makedirs(processed_data_folder, exist_ok=True)
os.makedirs(table_folder, exist_ok=True)

dataset_path = os.path.join(raw_data_folder, "European_Bank.csv")

print("Project folder:", project_folder)
print("Dataset path:", dataset_path)


# Load the original CSV dataset
if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        "Dataset not found. Please check that European_Bank.csv "
        "is inside the data/raw folder."
    )

df = pd.read_csv(dataset_path)

print("\nDataset loaded successfully.")


# Display dataset shape
print("\nDataset Shape")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])


# Display first five and last five records
print("\nFirst Five Records")
display(df.head())

print("\nLast Five Records")
display(df.tail())


# Check column names
print("\nColumn Names")
print(df.columns.tolist())


# Check data types
print("\nColumn Data Types")
display(df.dtypes.to_frame(name="Data Type"))


# Generate dataset information summary
print("\nDataset Information")
df.info()


# Check missing values
print("\nMissing Values")
missing_values = df.isnull().sum()
display(missing_values.to_frame(name="Missing Count"))


# Calculate missing-value percentages
print("\nMissing Value Percentages")
missing_percentage = (
    df.isnull().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percentage
})

display(missing_summary)


# Check complete duplicate records
complete_duplicates = df.duplicated().sum()

print("\nComplete Duplicate Records:", complete_duplicates)


# Check duplicate Customer IDs
if "CustomerId" in df.columns:
    duplicate_customer_ids = df["CustomerId"].duplicated().sum()

    print("Duplicate Customer IDs:", duplicate_customer_ids)
else:
    duplicate_customer_ids = None
    print("CustomerId column not found.")


# Validate numerical columns
print("\nNumerical Column Validation")

expected_numerical_columns = [
    "CustomerId",
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "EstimatedSalary",
    "Year"
]

numerical_validation = []

for column in expected_numerical_columns:
    if column in df.columns:

        converted_values = pd.to_numeric(df[column], errors="coerce")

        invalid_numeric_count = (
            converted_values.isna() & df[column].notna()
        ).sum()

        numerical_validation.append({
            "Column": column,
            "Original Data Type": str(df[column].dtype),
            "Invalid Numeric Values": invalid_numeric_count,
            "Minimum Value": converted_values.min(),
            "Maximum Value": converted_values.max()
        })

numerical_validation_df = pd.DataFrame(numerical_validation)

display(numerical_validation_df)


# Validate categorical columns
print("\nCategorical Column Validation")

expected_categorical_columns = [
    "Surname",
    "Geography",
    "Gender"
]

categorical_validation = []

for column in expected_categorical_columns:
    if column in df.columns:

        categorical_validation.append({
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Unique Values": df[column].nunique(dropna=True),
            "Missing Values": df[column].isnull().sum()
        })

categorical_validation_df = pd.DataFrame(categorical_validation)

display(categorical_validation_df)


# Validate binary columns
print("\nBinary Column Validation")

expected_binary_columns = [
    "HasCrCard",
    "IsActiveMember",
    "Exited"
]

binary_validation = []

for column in expected_binary_columns:
    if column in df.columns:

        values = pd.to_numeric(df[column], errors="coerce")

        invalid_binary_count = (
            values.notna() & ~values.isin([0, 1])
        ).sum()

        binary_validation.append({
            "Column": column,
            "Unique Values": df[column].dropna().unique().tolist(),
            "Invalid Binary Values": invalid_binary_count,
            "Missing Values": df[column].isnull().sum()
        })

binary_validation_df = pd.DataFrame(binary_validation)

display(binary_validation_df)


# Check numerical ranges and invalid values
print("\nNumerical Range Validation")

range_validation = []

range_rules = {
    "CreditScore": (0, None),
    "Age": (18, 120),
    "Tenure": (0, None),
    "Balance": (0, None),
    "NumOfProducts": (1, None),
    "EstimatedSalary": (0, None)
}

for column, (minimum_allowed, maximum_allowed) in range_rules.items():

    if column in df.columns:

        values = pd.to_numeric(df[column], errors="coerce")

        invalid_count = 0

        if minimum_allowed is not None:
            invalid_count += (values.notna() & (values < minimum_allowed)).sum()

        if maximum_allowed is not None:
            invalid_count += (values.notna() & (values > maximum_allowed)).sum()

        range_validation.append({
            "Column": column,
            "Minimum Allowed": minimum_allowed,
            "Maximum Allowed": maximum_allowed,
            "Observed Minimum": values.min(),
            "Observed Maximum": values.max(),
            "Invalid Range Values": invalid_count
        })

range_validation_df = pd.DataFrame(range_validation)

display(range_validation_df)


# Check target variable Exited distribution
print("\nTarget Variable Distribution")

if "Exited" in df.columns:

    exited_distribution = df["Exited"].value_counts(dropna=False).to_frame(
        name="Customer Count"
    )

    exited_distribution["Percentage"] = (
        df["Exited"].value_counts(normalize=True, dropna=False) * 100
    ).round(2)

    display(exited_distribution)

else:
    exited_distribution = pd.DataFrame()
    print("Exited column not found.")


# Check Year distribution, if the column exists
print("\nYear Distribution")

if "Year" in df.columns:

    year_distribution = df["Year"].value_counts(
        dropna=False
    ).sort_index().to_frame(name="Customer Count")

    year_distribution["Percentage"] = (
        df["Year"].value_counts(normalize=True, dropna=False) * 100
    ).round(2)

    display(year_distribution)

else:
    year_distribution = pd.DataFrame()
    print("Year column does not exist in this dataset.")


# Generate overall data-quality summary
print("\nOverall Data Quality Summary")

quality_summary = pd.DataFrame({
    "Validation Check": [
        "Total Records",
        "Total Columns",
        "Total Missing Values",
        "Complete Duplicate Records",
        "Duplicate Customer IDs",
        "Columns with Missing Values",
        "Columns with Invalid Numeric Values",
        "Columns with Invalid Binary Values",
        "Columns with Invalid Range Values"
    ],
    "Result": [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(complete_duplicates),
        duplicate_customer_ids,
        int((df.isnull().sum() > 0).sum()),
        int(
            (numerical_validation_df["Invalid Numeric Values"] > 0).sum()
        ) if not numerical_validation_df.empty else 0,
        int(
            (binary_validation_df["Invalid Binary Values"] > 0).sum()
        ) if not binary_validation_df.empty else 0,
        int(
            (range_validation_df["Invalid Range Values"] > 0).sum()
        ) if not range_validation_df.empty else 0
    ]
})

display(quality_summary)


# Save validation results and original dataset copy
original_copy_path = os.path.join(
    processed_data_folder,
    "European_Bank_Original_Copy.csv"
)

missing_report_path = os.path.join(
    table_folder,
    "Missing_Values_Report.csv"
)

numerical_report_path = os.path.join(
    table_folder,
    "Numerical_Validation_Report.csv"
)

categorical_report_path = os.path.join(
    table_folder,
    "Categorical_Validation_Report.csv"
)

binary_report_path = os.path.join(
    table_folder,
    "Binary_Validation_Report.csv"
)

range_report_path = os.path.join(
    table_folder,
    "Numerical_Range_Validation_Report.csv"
)

quality_report_path = os.path.join(
    table_folder,
    "Data_Quality_Summary.csv"
)

# Preserve a separate copy of the original dataset
shutil.copy2(dataset_path, original_copy_path)

# Save validation reports
missing_summary.to_csv(missing_report_path)

numerical_validation_df.to_csv(
    numerical_report_path, index=False
)

categorical_validation_df.to_csv(
    categorical_report_path, index=False
)

binary_validation_df.to_csv(
    binary_report_path, index=False
)

range_validation_df.to_csv(
    range_report_path, index=False
)

quality_summary.to_csv(
    quality_report_path, index=False
)

print("\nAll validation reports saved successfully.")

print("\nOriginal Dataset Copy:")
print(original_copy_path)

print("\nValidation Reports Folder:")
print(table_folder)

print("\nData ingestion and validation completed successfully.")

Project folder: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy
Dataset path: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\raw\European_Bank.csv

Dataset loaded successfully.

Dataset Shape
Number of rows: 10000
Number of columns: 14

First Five Records


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0



Last Five Records


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9995,2025,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,2025,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,2025,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,2025,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1
9999,2025,15628319,Walker,792,France,Female,28,4,130142.79,1,1,0,38190.78,0



Column Names
['Year', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']

Column Data Types


,Data Type
Year,int64
CustomerId,int64
Surname,object
CreditScore,int64
Geography,object
Gender,object
Age,int64
Tenure,int64
Balance,float64
NumOfProducts,int64


... (15 outputs left out)

,Customer Count,Percentage
Year,,
2025,10000,100.00



Overall Data Quality Summary


,Validation Check,Result
0,Total Records,10000
1,Total Columns,14
2,Total Missing Values,0
3,Complete Duplicate Records,0
4,Duplicate Customer IDs,0
5,Columns with Missing Values,0
6,Columns with Invalid Numeric Values,0
7,Columns with Invalid Binary Values,0
8,Columns with Invalid Range Values,0



All validation reports saved successfully.

Original Dataset Copy:
C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\European_Bank_Original_Copy.csv

Validation Reports Folder:
C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\outputs\tables

Data ingestion and validation completed successfully.
